In [1]:
from torchvision.datasets import FashionMNIST

from dataeval_flow.config import (
    BoVWExtractorConfig,
    DataCleaningWorkflowConfig,
    DatasetProtocolConfig,
    PipelineConfig,
    SourceConfig,
    TaskConfig,
    ViewConfig,
    ViewOperation,
)
from dataeval_flow.workflow import run_tasks

# 1. Create the torchvision dataset (no transforms — the adapter handles conversion)
tv_dataset = FashionMNIST(root="./data", train=True, download=True)

  0%|          | 0.00/26.4M [00:00<?, ?B/s]

  0%|          | 32.8k/26.4M [00:00<01:57, 224kB/s]

  0%|          | 65.5k/26.4M [00:00<01:58, 222kB/s]

  0%|          | 131k/26.4M [00:00<01:21, 322kB/s] 

  1%|          | 229k/26.4M [00:00<00:57, 457kB/s]

...

 41%|████▏     | 1.84M/4.42M [00:01<00:00, 3.13MB/s]

 83%|████████▎ | 3.67M/4.42M [00:01<00:00, 6.11MB/s]

100%|██████████| 4.42M/4.42M [00:01<00:00, 3.75MB/s]

  0%|          | 0.00/5.15k [00:00<?, ?B/s]

100%|██████████| 5.15k/5.15k [00:00<00:00, 13.3MB/s]

In [2]:
# 2. Build the full pipeline config (using a small subset for speed)
datasets = [DatasetProtocolConfig(name="fmnist-train", format="torchvision", dataset=tv_dataset)]
views = [ViewConfig(name="first500", operations=[ViewOperation(type="Limit", params={"size": 500})])]
sources = [SourceConfig(name="fmnist-src", dataset="fmnist-train", view="first500")]
extractors = [BoVWExtractorConfig(name="bovw", vocab_size=512, batch_size=64)]

workflows = [
    DataCleaningWorkflowConfig(
        name="adaptive_clean",
        outlier_method="adaptive",
        outlier_threshold=3.5,
        outlier_flags=["dimension", "pixel", "visual"],
    )
]
tasks = [
    TaskConfig(
        name="fmnist-clean",
        workflow="adaptive_clean",
        sources="fmnist-src",
        extractor="bovw",
    )
]

config = PipelineConfig(
    datasets=datasets,
    views=views,
    sources=sources,
    extractors=extractors,
    workflows=workflows,
    tasks=tasks,
)

In [3]:
# 3. Run
results = run_tasks(config)
print(results[0].report())


  DATA CLEANING COMPLETE. DATASET: 500 ITEMS. MODE: ADVISORY.
  Timestamp:    2026-09-04T07:25:22.384024+00:00
  Duration:     1.00s
  Source:       fmnist-src (fmnist-train[first500])
  Model:        bovw (bovw)
--------------------------------------------------------------------------------

  SUMMARY
  -------
  Image Outliers ....................................... 6 images (1.2%)  [..]
  Classwise Outliers ....... worst: Sandal (3.9%), 1/5 classes over 3.0%  [..]
  Duplicates ............................ 0 exact (0.0%), 10 near (2.0%)  [..]
  Label Distribution ............ 10 classes, 500 items, imbalance 1.3:1  [..]

  Health: All checks passed [ok]

  IMAGE OUTLIERS                                                 6 images (1.2%)
  6 images (1.2%) flagged as outliers.

  Metric      Count
  ----------  -----
  kurtosis        5
  sharpness       2
  skew            2
  brightness      1

  (Some images trigger multiple metrics.)

  percentage    1.2
  dataset_size  500

  CLASS

/builds/jatic/aria/dataeval-flow/src/dataeval_flow/binning.py:302: UserWarning: `id` was binned automatically ('uniform_width') because no bins were declared. The bin count is derived from the data, so it is not stable across samples and the same factor measured twice may not be comparable. Declare cutoffs with continuous_factor_bins={"id": [...]} to control this.
  factor_info = metadata.factor_info


In [4]:
DatasetProtocolConfig(
    name="fmnist-train",
    format="torchvision",
    dataset=tv_dataset,
    version="2",  # bump this when the underlying data changes
)

DatasetProtocolConfig(name='fmnist-train', format='torchvision', dataset=Dataset FashionMNIST
    Number of datapoints: 60000
    Root location: ./data
    Split: Train, version='2')